# Глава 7. Деревья решений: компьютер-детектив

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь объяснить, как дерево решений делит данные вопросами, построить и визуализировать дерево через plot_tree, а также распознать переобучение и ограничить его параметром max_depth.

KNN сравнивал объекты по похожести. Дерево решений действует совершенно иначе — оно задаёт данным вопросы, шаг за шагом сужая круг, как детектив на допросе. И главное его достоинство: решение дерева можно нарисовать и понять глазами, в отличие от многих других моделей.

**Аналогия: игра «20 вопросов»  💡**

Ты загадал персонажа, а друг угадывает через вопросы с ответом «да/нет»: «Это человек? Он реальный? Он спортсмен?» Каждый хороший вопрос отсекает половину вариантов и приближает к ответу. Дерево решений делает ровно это: задаёт признакам вопросы вроде «Длина клюва < 45 мм?», «Масса < 4000 г?» — и с каждым ответом приближается к предсказанию.

## Узлы, ветви и листья

Дерево состоит из трёх частей. Узлы — в каждом стоит вопрос по одному признаку. Ветви — это ответы «да»/«нет», по которым мы движемся дальше. Листья — конечные точки с финальным предсказанием. Начинаем сверху (корень), отвечаем на вопросы, спускаемся по ветвям и приходим в лист с ответом.

Как дерево выбирает, какой вопрос задать первым? Оно ищет тот вопрос, который лучше всего разделяет данные на «чистые» группы — где в одну сторону уходят в основном пингвины одного вида, в другую — другого. Это называется информативностью вопроса: хороший вопрос сильно уменьшает неопределённость.

## Игра «20 вопросов» с рисованием

Сыграйте в классе в «20 вопросов», но рисуйте дерево на доске: каждый заданный вопрос — узел, каждый ответ — ветвь. Через несколько раундов вы получите настоящее дерево решений, построенное руками. Это лучший способ прочувствовать, как алгоритм движется от общих вопросов к конкретному ответу.

## Дерево в коде и его картинка

Построим дерево на пингвинах и — самое интересное — нарисуем его. Возможность увидеть логику модели целиком делает деревья незаменимыми для обучения и объяснения решений.

In [ ]:
# === Подготовка: выполни первой, если открыл новый блокнот ===
import pandas as pd
from sklearn.model_selection import train_test_split
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url).dropna(subset=["bill_length_mm", "body_mass_g", "species"])
X = df[["bill_length_mm", "body_mass_g"]]
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

# max_depth=3 — ограничиваем глубину, чтобы дерево было читаемым
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

# Рисуем дерево
plt.figure(figsize=(14, 6))
plot_tree(tree, feature_names=["bill_length_mm", "body_mass_g"],
          class_names=["Adelie", "Chinstrap", "Gentoo"], filled=True)
plt.show()

_Ожидаемый результат: схема дерева с вопросами в узлах и предсказаниями в листьях._

## Переобучение и параметр max_depth

**Аналогия: зубрёжка каждой мелочи  💡**

Если дереву не ограничивать глубину, оно будет задавать вопросы, пока не запомнит каждый обучающий пример по отдельности — вплоть до «этому конкретному пингвину ставим Adelie». На train будет 100%, а на новых данных — провал. Это переобучение (overfitting): модель выучила случайные частности вместо общих закономерностей. Как ученик, зазубривший конкретные задачи вместо понимания темы.

Есть и противоположная беда — недообучение (underfitting): дерево слишком простое (например, глубина 1) и не улавливает даже явных закономерностей. Задача — найти баланс между этими крайностями. Главный рычаг здесь — max_depth, максимальная глубина дерева: чем она больше, тем сложнее модель и тем выше риск переобучения.

In [ ]:
# Сравним дерево без ограничения и с max_depth=3
for depth in [1, 3, 5, None]:   # None = без ограничения
    t = DecisionTreeClassifier(max_depth=depth, random_state=42)
    t.fit(X_train, y_train)
    print(f"depth={str(depth):4}: "
          f"train={t.score(X_train, y_train):.2f}, "
          f"test={t.score(X_test, y_test):.2f}")

Ты увидишь закономерность: с ростом глубины train-accuracy ползёт вверх (вплоть до ~1,00 без ограничения), а test-accuracy сначала растёт, потом останавливается или падает. Момент, когда train высоко, а test низко — это и есть переобучение, наглядно в числах. Оптимальная глубина — там, где test максимален.

## Минута этики ML

Дерево наглядно показывает, по каким признакам оно принимает решения. А что, если модель решает важные вещи о людях — по признаку пола, возраста, места жительства? Это повод для серьёзного разговора: какие признаки использовать этично, а какие могут закрепить несправедливость. Например, модель одобрения кредита, обученная на предвзятых исторических данных, может унаследовать эту предвзятость. Модель делает лишь то, чему её научили данные, — а ответственность за последствия лежит на человеке.

## Задания

Базовый уровень

Построй и нарисуй дерево с max_depth=3 на пингвинах. Прочитай картинку: какой вопрос дерево задаёт первым (в корне)? Какой признак оно считает самым важным?

Со звёздочкой  ⭐

Запусти цикл сравнения глубин. Найди глубину, при которой test-accuracy максимальна. Затем поставь заведомо большую глубину (например, 20) и убедись в переобучении: train близок к 1,00, а test ниже. Запиши, при какой глубине начинается переобучение.

Вопросы для обсуждения  💬

Чем дерево решений удобнее KNN для объяснения решения человеку?

Как по значениям train и test понять, что модель переобучилась?

Приведи пример задачи, где использование некоторых признаков (пол, возраст) было бы этически спорным.

**Проверь себя  ℹ️**

1. Из чего состоит дерево решений (узлы, ветви, листья)?

2. Что такое переобучение и как его заметить?

3. Какой параметр ограничивает сложность дерева?